In [1]:
import os
import json
import platform
from bs4 import BeautifulSoup
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
load_dotenv(override = 'True')
api_key = os.getenv('OPEN_AI_KEY')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [35]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    def __init__(self, url, use_selenium=False):
        self.url = url

        if use_selenium:
            self.body = self._fetch_with_selenium()
        else:
            self.body = self._fetch_with_requests()

        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""

        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def _fetch_with_requests(self):
        try:
            response = requests.get(self.url, headers=headers, timeout=10)
            response.raise_for_status()
            return response.content
        except requests.RequestException as e:
            print(f"Request error: {e}")
            return ""

    def _fetch_with_selenium(self):
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1200")

        # Attempt to detect platform and assign fallback binary path
        system = platform.system()
        chrome_paths = {
            "Darwin": "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",  # macOS
            "Linux": "/usr/bin/google-chrome",
            "Windows": "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"
        }

        fallback_path = chrome_paths.get(system)
        if fallback_path and os.path.exists(fallback_path):
            options.binary_location = fallback_path
        else:
            print("⚠️ Warning: Chrome binary not found at default path. If you have Chrome installed elsewhere, set binary_location manually.")

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

        try:
            driver.get(self.url)
            html = driver.page_source
        finally:
            driver.quit()
        return html
    def get_contents(self):
        return {
             "title" : self.title,
             "text": self.text,
             "links" : self.links
        }


In [36]:
link_system_prompt = """
You are provided with a list of hyperlinks extracted from a person's personal website. 
Your task is to decide which of these links are most relevant for building a resume or CV about the person.

Relevant links typically include (but are not limited to):
- About page
- Awards
- Publications
- Talks / Presentations / Conferences
- Grants / Research Projects
- Professional Profiles (LinkedIn, Google Scholar, etc.)

Ignore links that point to unrelated content (e.g., Careers, Terms of Service, Privacy Policy, External Advertisements).

Respond strictly in the following JSON format:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "publications", "url": "https://another.full.url/publications"},
        {"type": "awards", "url": "https://full.url/awards"}
    ]
}

Do not include any explanation text — only return the JSON object.
"""

In [37]:
def get_links(url):
    website = Website(url, use_selenium=True)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [38]:
from urllib.parse import urljoin

def get_links_user_prompt(website):
    full_links = [urljoin(website.url, link) for link in website.links]
    full_links = list(set(full_links))  # Remove duplicates
    full_links.sort()
    max_links = 50
    display_links = full_links[:max_links]

    user_prompt = f"You are given the following list of links found on the website: {website.url}.\n"
    user_prompt += "Your task is to identify which of these links are relevant for including in a resume or CV about the person.\n\n"
    user_prompt += "Relevant links may include:\n"
    user_prompt += "- About page\n"
    user_prompt += "- Awards / Honors\n"
    user_prompt += "- Publications\n"
    user_prompt += "- Talks / Presentations / Conferences\n"
    user_prompt += "- Grants / Research Projects\n"
    user_prompt += "- Professional profiles (LinkedIn, Google Scholar, etc.)\n\n"
    user_prompt += "Ignore unrelated links (e.g., Careers page, Privacy Policy, Terms, Contact forms).\n"
    user_prompt += "Respond strictly in JSON format as shown in this example:\n\n"
    user_prompt += """{
    "links": [
        {"type": "about page", "url": "https://full.url/about"},
        {"type": "publications", "url": "https://full.url/publications"}
    ]
}
"""
    user_prompt += "\nHere are the links (up to 50):\n"
    user_prompt += "\n".join(display_links)

    return user_prompt   

In [39]:
get_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'publications',
   'url': 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/'},
  {'type': 'publications',
   'url': 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/'},
  {'type': 'publications',
   'url': 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/'},
  {'type': 'publications',
   'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'patents',
   'url': 'https://patents.google.com/patent/US20210049536A1/'}]}

In [51]:
def get_all_details(url):
    result = "Landing page:\n"
    main_site = Website(url, use_selenium=True)
    result += main_site.get_contents()["text"] + "\n"
    
    links = get_links(url)
    print("Found links:", links)
    
    for link in links.get("links", []):
        link_type = link.get("type")
        link_url = link.get("url")
        if not link_url:
            continue 
        try: 
            linked_site = Website(link_url, use_selenium=True)
            result += f"\n\n## {link_type}\n"
            result += linked_site.get_contents()["text"]
        except Exception as e:
            print(f"Error processing {links}: {e}")

    return result

In [52]:
system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a person's website 
(e.g., About page, Publications, Awards, Talks, Grants, and similar sections).

Your task is to:
1. Generate a concise, well-structured CV or resume for recruiters, based on the provided content.
2. Summarize key achievements, roles, publications, awards, and other notable information.
3. Suggest an appropriate job title that best fits the person, based on their expertise and achievements.

The response should be formatted in **Markdown**, using clear section headings such as:
- About / Summary
- Education
- Awards and Honors
- Publications
- Talks / Presentations
- Grants / Research Projects
- Suggested Job Title

Focus on **clarity, brevity, and relevance** for recruiters.

Do not fabricate information. Only use the data provided.
"""


In [53]:
def get_cv_user_prompt(full_name, url):
    user_prompt = f"You are analyzing the website of {full_name} ({url}).\n\n"
    user_prompt += "Below is the content from the landing page and other relevant sections of the site:\n\n"
        
    website_details = get_all_details(url)
    website_details = website_details[:5000]  # Truncate 
    user_prompt += website_details
    
    return user_prompt


In [54]:
def Bulid_CV(full_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_cv_user_prompt(full_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [55]:
Bulid_CV("Ed Donner", "https://edwarddonner.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}, {'type': 'professional profile', 'url': 'https://www.linkedin.com/in/eddonner/'}, {'type': 'patents', 'url': 'https://patents.google.com/patent/US20210049536A1/'}]}


# Ed Donner's CV

## About / Summary
Co-Founder and CTO of Nebula.io, dedicated to innovating talent sourcing and management through Generative AI and machine learning. Former CEO of AI startup untapt, which was acquired in 2021. Passionate about applying AI to solve real-world problems and enhancing employee engagement and fulfillment through better job alignment.

## Education
(Not Provided)

## Awards and Honors
- **American Banker Top 20 Company To Watch**: Recognition for untapt during its operational period.
- **Voted 'Startup Most Likely to Grow Exponentially'**: At an Amazon pitch event, highlighting untapt’s potential.

## Publications
- **The Complete Agentic AI Engineering Course** (2025)
- **LLM Workshop – Hands-on with Agents** (2025)
- **Mastering AI and LLM Engineering** (2024)

## Talks / Presentations
- Featured interviews on the floor of the New York Stock Exchange and Nasdaq discussing recruitment innovations.

## Grants / Research Projects
- Participated in the **Accenture FinTech Innovation Lab** for untapt.

## Suggested Job Title
**Chief Technology Officer (CTO) & AI Innovation Leader**

In [57]:
def stream_cv(full_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_cv_user_prompt(full_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [58]:
stream_cv("Ed Donner", "https://edwarddonner.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/'}, {'type': 'publications', 'url': 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/'}, {'type': 'professional profile', 'url': 'https://www.linkedin.com/in/eddonner/'}, {'type': 'patents', 'url': 'https://patents.google.com/patent/US20210049536A1/'}]}


# Ed Donner's CV

## About / Summary
Ed Donner is the co-founder and CTO of Nebula.io, where he utilizes Generative AI and machine learning to revolutionize talent sourcing and recruitment engagement. With a strong background in software engineering, data science, and technology leadership, Ed has dedicated his career to solving complex hiring challenges, striving to help individuals discover their potential and achieve fulfillment in their professional lives. Previously, he founded untapt, an AI startup acquired in 2021, which built innovative recruitment marketplaces.

## Education
*Details not provided*

## Awards and Honors
- American Banker Top 20 Companies to Watch
- Featured in Fast Company, Forbes, and American Banker

## Publications
- "The Complete Agentic AI Engineering Course" (April 2025)
- "LLM Workshop – Hands-on with Agents" (January 2025)
- "Mastering AI and LLM Engineering" (November 2024)

## Talks / Presentations
- Interviewed on the floor of the New York Stock Exchange and Nasdaq about the impact of AI in recruitment.
  
## Grants / Research Projects
*Details not provided*

## Suggested Job Title
**Chief Technology Officer (CTO) and AI Solutions Architect**

In [59]:
stream_cv("Elon Musk", "https://en.wikipedia.org/wiki/Elon_Musk")

Found links: {'links': [{'type': 'about page', 'url': 'https://en.wikipedia.org/wiki/Elon_Musk'}, {'type': 'awards', 'url': 'http://www.inc.com/magazine/20071201/entrepreneur-of-the-year-elon-musk.html'}, {'type': 'publications', 'url': 'https://archive.today/20230909163003/https://www.nytimes.com/2023/09/09/books/review/elon-musk-walter-isaacson.html'}, {'type': 'talks/presentations/conferences', 'url': 'http://media.aerosociety.com/aerospace-insight/2012/11/23/video-elon-musk-interview/7553'}, {'type': 'talks/presentations/conferences', 'url': 'http://www.businessweek.com/articles/2013-08-12/revealed-elon-musk-explains-the-hyperloop'}]}


# Elon Musk CV

## About / Summary
Elon Musk is a prominent entrepreneur and business magnate known for founding and leading several groundbreaking companies in electric vehicles, space exploration, and technology. As the CEO and product architect of Tesla and the founder of SpaceX, he has played a crucial role in shaping modern technology and sustainability. Musk also has a diverse portfolio that includes ventures in artificial intelligence, tunnel construction, and high-speed transportation systems. 

## Education
- **University of Pennsylvania**
  - Bachelor of Arts (BA)
  - Bachelor of Science (BS)

## Awards and Honors
- Accolades from various industries for innovation and leadership. 
- Numerous recognitions for contributions to technology and sustainability sectors.

## Publications
- No specific publications listed; however, Musk's ventures have inspired a wealth of literature and media coverage focusing on technology, innovation, and business strategies.

## Talks / Presentations
- Renowned speaker on topics related to space exploration, sustainability, and technology innovation. Notable speeches include discussions on Tesla developments and SpaceX missions.

## Grants / Research Projects
- Active in funding and supporting initiatives through the Musk Foundation focusing on renewable energy, education, and space exploration.

## Suggested Job Title
**CEO and Chief Engineer of Innovative Technological Ventures** 
